In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
import os

In [ ]:
# 1. tool create
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 number a and b and this tool return their product."""
    return a * b

In [5]:
print(multiply.invoke({'a': 4, 'b': 5}))

20


In [6]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 number a and b and this tool return their product.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [ ]:

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=(
        os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
    )
)

model = ChatHuggingFace(llm=llm)

In [26]:
model.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 30, 'total_tokens': 40}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d534c-14de-70e0-bdfa-32aa00a96e4f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 10, 'total_tokens': 40})

In [ ]:
# 2. tool binding
llm_with_tools = model.bind_tools([multiply])

In [28]:
llm_with_tools.invoke("hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 213, 'total_tokens': 223}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d534c-1e22-7060-bc03-7ea5acedf76b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 10, 'total_tokens': 223})

In [40]:
query = HumanMessage("can you multiply 3 with 99")

In [41]:
messages = [query]

In [42]:
messages

[HumanMessage(content='can you multiply 3 with 99', additional_kwargs={}, response_metadata={})]

In [43]:
result = llm_with_tools.invoke(messages)

In [44]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 99},
 'id': 'call_i9b9irrwo3g7f0uq3g373ub1',
 'type': 'tool_call'}

In [45]:
tool_result = multiply.invoke(result.tool_calls[0])

In [46]:
messages.append(result)

In [47]:
messages

[HumanMessage(content='can you multiply 3 with 99', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":99}', 'name': 'multiply', 'description': None}, 'id': 'call_i9b9irrwo3g7f0uq3g373ub1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 221, 'total_tokens': 247}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5352-2b82-76e0-aa00-f76cb72e0a75-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 99}, 'id': 'call_i9b9irrwo3g7f0uq3g373ub1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 26, 'total_tokens': 247})]

In [48]:
messages.append(tool_result)

In [49]:
messages

[HumanMessage(content='can you multiply 3 with 99', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":99}', 'name': 'multiply', 'description': None}, 'id': 'call_i9b9irrwo3g7f0uq3g373ub1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 221, 'total_tokens': 247}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5352-2b82-76e0-aa00-f76cb72e0a75-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 99}, 'id': 'call_i9b9irrwo3g7f0uq3g373ub1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 26, 'total_tokens': 247}),
 ToolMessage(content='297', name='multiply', tool_call_id='call_i9b9irrwo3g7f0uq3g373ub1')]

In [50]:
llm_with_tools.invoke(messages).content

'The product of 3 multiplied by 99 is 297.'